In [1]:
# Definir constantes
PKL_MODEL = "health.pkl"
PRODUCT = "health"
START_DATE = "2025-01-01"
END_DATE = "2025-12-01"
BUDGET = 1896.34
PCT_SPEND = [0.0892,0.022,0.071,0,0.6923,0,0.1153,0.0102]
LOWER_RULES = {
    "off-television": 0,
    "off-radio": 0,
    "off-outdoor": 0,
    "off-news": 0,
}
UPPER_RULES = {
    "off-television": 0,
    "off-radio": 0,
    "off-outdoor": 0,
    "off-news": 0,
}
SPEND_RULES_HARDCODED = {}

In [2]:
from meridian.analysis import optimizer
import joblib
import pandas as pd

mmm = joblib.load(PKL_MODEL)

In [3]:
channels = mmm.input_data.get_all_paid_channels()
lower_constraints = []
upper_constraints = []

for ch in channels:
  if ch in LOWER_RULES:
    lower_constraints.append(LOWER_RULES[ch])
  else:
    lower_constraints.append(0.5)
  if ch in UPPER_RULES:
    upper_constraints.append(UPPER_RULES[ch])
  else:
    upper_constraints.append(0.5)

In [4]:
# ========================================
# OPTIMIZACIÓN PRESUPUESTARIA 1
# ========================================
budget_optimizer = optimizer.BudgetOptimizer(mmm)

optimization_results = budget_optimizer.optimize(
    use_posterior=True,
    selected_times=None,
    fixed_budget=True,
    budget=None,
    start_date=START_DATE,
    end_date=END_DATE,
    spend_constraint_lower=lower_constraints,
    spend_constraint_upper=upper_constraints,
    target_roi=None,
    target_mroi=None,
    gtol=0.0001,
    use_optimal_frequency=True,
    use_kpi=True,
    confidence_level=0.9,
    batch_size=3000,
)

df_nonopt = optimization_results._get_delta_data(
    metric="incremental_outcome",
    metric_int="nonopt",
)

df_nonopt_spend = optimization_results._get_delta_data(
    metric="spend",
    metric_int="nonopt",
)

In [5]:
# ========================================
# OPTIMIZACIÓN PRESUPUESTARIA 2
# ========================================
budget_optimizer = optimizer.BudgetOptimizer(mmm)

optimization_results = budget_optimizer.optimize(
    use_posterior=True,
    selected_times=None,
    fixed_budget=True,
    budget=BUDGET,
    start_date=START_DATE,
    end_date=END_DATE,
    pct_of_spend=PCT_SPEND,
    spend_constraint_lower=lower_constraints,
    spend_constraint_upper=upper_constraints,
    target_roi=None,
    target_mroi=None,
    gtol=0.0001,
    use_optimal_frequency=True,
    use_kpi=True,
    confidence_level=0.9,
    batch_size=3000,
)

df_opt = optimization_results._get_delta_data(
    metric="incremental_outcome",
    metric_int="opt",
)

df_opt_spend = optimization_results._get_delta_data(
    metric="spend",
    metric_int="opt",
)

In [6]:
# df final incremental_outcome
df_final = pd.merge(
    df_opt, df_nonopt, on='channel', suffixes=('_opt', '_nonopt')
)

metric_opt = 'incremental_outcome_opt'
metric_nonopt = 'incremental_outcome_nonopt'
df_final['diff'] = df_final[metric_opt] - df_final[metric_nonopt]

data = {
    item['channel']: round(item['diff'], 6)
    for item in df_final.to_dict(orient='records')
}
data['optimized'] = 0.0
data['non_optimized'] = float(round(df_final[metric_nonopt].sum(), 6))


# Sort the dictionary
first = {'non_optimized': data.pop('non_optimized')}
last = {'optimized': data.pop('optimized')}

negative = {k: v for k, v in data.items() if v < 0}
positive = {k: v for k, v in data.items() if v > 0}
zero = {k: v for k, v in data.items() if v == 0}

negative_sorted = dict(sorted(negative.items(), key=lambda x: x[1]))
positive_sorted = dict(
    sorted(positive.items(), key=lambda x: x[1], reverse=True)
)
zero_sorted = dict(zero.items())

dict_final = {
    **first,
    **negative_sorted,
    **positive_sorted,
    **zero_sorted,
    **last,
}

In [7]:
# df final spend
df_final_spend = pd.merge(
    df_opt_spend, df_nonopt_spend, on='channel', suffixes=('_opt', '_nonopt')
)

metric_opt = 'spend_opt'
metric_nonopt = 'spend_nonopt'
df_final_spend['diff'] = (
    df_final_spend[metric_opt] - df_final_spend[metric_nonopt]
)

data = {
    item['channel']: round(item['diff'], 6)
    for item in df_final_spend.to_dict(orient='records')
}

for ch_ in data.keys():
  if ch_ in SPEND_RULES_HARDCODED.keys():
    data[ch_] = SPEND_RULES_HARDCODED[ch_]

negative = {k: v for k, v in data.items() if v < 0}
positive = {k: v for k, v in data.items() if v > 0}
zero = {k: v for k, v in data.items() if v == 0}

negative_sorted = dict(sorted(negative.items(), key=lambda x: x[1]))
positive_sorted = dict(
    sorted(positive.items(), key=lambda x: x[1], reverse=True)
)
zero_sorted = dict(zero.items())

dict_final_spend = {**negative_sorted, **positive_sorted, **zero_sorted}

In [8]:
import math

non_optimized = round(dict_final["non_optimized"], 1)
optimized = round(sum(dict_final.values()), 1)

domain_min = math.floor(non_optimized / 100) * 100 - 300
domain_max = math.ceil(optimized / 100) * 100 + 100

output_filename = f"incremental_outcome_delta_{PRODUCT}.png"
chart = optimization_results.plot_incremental_outcome_delta(
    df_dict=dict_final, custom_d_e=(domain_min, domain_max)
)
chart.save(output_filename)
print(f"non-optimized: {non_optimized}")
print(f"optimized: {optimized}")
chart

non-optimized: 6622.3
optimized: 6926.6


alt.LayerChart(...)

In [9]:
output_filename = f"incremental_spend_delta_{PRODUCT}.png"
chart = optimization_results.plot_spend_delta(
    df_dict=dict_final_spend
)
chart.save(output_filename)
chart

alt.LayerChart(...)